In [4]:
!pip install dataset

# o1 推理数据处理

xiaodongguaAIGC

各个数据集特点：

1. GSM8K: 小学数学题, 有step，但都是正确的
2. PRM800K: 有step，prompt有大量重复，错误step质量差，部分题目有难度，有部分解答不完整，部分数据带反思
3. MATH：无STEP，竞赛难题，
4. ProcessBench： Qwen采样多个模型输出，并标注首个错误步骤，标注不完整。有step解答

数据主要分两种
1. STEP SFT， 主要用正确步骤数据训练，把包含错误step的数据进行过滤，PRM800K所训练的模型具有反思风格
2. PRM， 主要需要有错误的Step


PRM or ORM
1. 部分数据，如MATH没有明显的step隔断，仅有ORM标签，那么实际训练混合数据后，可以进行PRM/ORM混合训练


我们的数据格式
1. type：表明来源
2. is_step: 标明解答步骤是否有list组成，而MATH整段的解答，我们当成一个step, 并说明非step数据
3. is_end：PRM800K有的解答不完整，在训练时，不能加EOS
4. prompt：问题
5. completions： 字符串列表
6. labels：解答步骤数量对应bool列表


数据加载：

同时，也可以直接下载：

1.  [xiaodongguaAIGC/step_sft](https://huggingface.co/datasets/xiaodongguaAIGC/step_sft)
2. [xiaodongguaAIGC/step_prm](https://huggingface.co/datasets/xiaodongguaAIGC/step_prm)

In [5]:
!pip install datasets

# Step-SFT Dataset

## PRM800K

In [6]:
from datasets import load_dataset, DatasetDict, Dataset

dataset = load_dataset("plaguss/prm_800k_trl",)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/2.71k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/50.1M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/389725 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10246 [00:00<?, ? examples/s]

In [7]:
dataset = dataset.remove_columns('index')

In [8]:
len(dataset['train'])
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completions', 'labels'],
        num_rows: 389725
    })
    test: Dataset({
        features: ['prompt', 'completions', 'labels'],
        num_rows: 10246
    })
})


In [19]:
def filter(dataset):
    data_list = []
    for data in dataset:
        l = len(data['completions'])
        answer_last_1_step = False
        answer_last_2_step = False
        if '\n\n# Answer\n\n' in data['completions'][-1]:
            answer_last_1_step = True
        if l >= 2 and '\n\n# Answer\n\n' in data['completions'][-2]:
            answer_last_2_step = True
            data['completions'] = data['completions'][:-1]
            data['labels'] = data['labels'][:-1]

        data['is_step'] = True
        data['type'] = 'PRM800K'
        if all(data['labels']) and (answer_last_1_step or  answer_last_2_step):
            data['is_end'] = True
            data_list.append(data)
        elif all(data['labels']):
            data['is_end'] = False
            data_list.append(data)

    # filter 重复 prompt
    data_only_list = []
    pre_string = 'null'
    is_new_prompt = True
    for data in data_list:
        if data['prompt'] == pre_string:
            continue
        else:
            data_only_list.append(data)
            pre_string = data['prompt']
    len(data_only_list)

    return data_only_list

In [20]:
new_dataset_train = filter(dataset['train'])
new_dataset_test = filter(dataset['test'])
print(len(new_dataset_train))

60504


In [21]:
def list_to_dict(list):
    dict = {'prompt':[], 'completions':[], 'labels': [], 'type': [], 'is_step': [], 'is_end': []}
    for item in list:
        dict['prompt'].append(item['prompt'])
        dict['completions'].append(item['completions'])
        dict['labels'].append(item['labels'])
        dict['type'].append(item['type'])
        dict['is_step'].append(item['is_step'])
        dict['is_end'].append(item['is_end'])
    return dict

In [22]:
hf_dataset_train = Dataset.from_dict(list_to_dict(new_dataset_train))
print(hf_dataset_train)
hf_dataset_test = Dataset.from_dict(list_to_dict(new_dataset_test))
print(hf_dataset_test)
prm800k_dataset = DatasetDict( {'train' : hf_dataset_train,
                'test' : hf_dataset_test})
print(prm800k_dataset)

Dataset({
    features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
    num_rows: 60504
})
Dataset({
    features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
    num_rows: 1227
})
DatasetDict({
    train: Dataset({
        features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
        num_rows: 60504
    })
    test: Dataset({
        features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
        num_rows: 1227
    })
})


## GSM8K

In [24]:
from datasets import load_dataset, DatasetDict, Dataset

dataset = load_dataset("openai/gsm8k",'main')
print(dataset)
print(dataset['train'][0]['answer']) # has /n

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


In [25]:
text = dataset['train'][0]['answer']
lines = text.split('\n')
print(lines)
labels = [True] * len(lines)
print(labels)

['Natalia sold 48/2 = <<48/2=24>>24 clips in May.', 'Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.', '#### 72']
[True, True, True]


In [26]:
def process_gsm8k(example):
  text = example['answer']
  lines = text.split('\n')
  labels = [True] * len(lines)
  example['prompt'] = example['question']

  lines[-1] = lines[-1].replace('#### ', 'Answer: ')
  example['completions'] = lines
  example['labels'] = labels
  example['is_end'] = True
  example['is_step'] = True
  example['type'] = 'GSM8K'
  return example

gsm8k_dataset = dataset.map(process_gsm8k, remove_columns=['question', 'answer'])

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [27]:
print(gsm8k_dataset)
print(gsm8k_dataset['train'][0]['completions']) # has /n

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completions', 'labels', 'is_end', 'is_step', 'type'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['prompt', 'completions', 'labels', 'is_end', 'is_step', 'type'],
        num_rows: 1319
    })
})
['Natalia sold 48/2 = <<48/2=24>>24 clips in May.', 'Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.', 'Answer: 72']


## MATH

In [28]:
!wget https://people.eecs.berkeley.edu/~hendrycks/MATH.tar
!tar -xvf MATH.tar

流式输出内容被截断，只能显示最后 5000 行内容。
MATH/test/algebra/2130.json
MATH/test/algebra/1659.json
MATH/test/algebra/1209.json
MATH/test/algebra/816.json
MATH/test/algebra/1570.json
MATH/test/algebra/1065.json
MATH/test/algebra/400.json
MATH/test/algebra/2649.json
MATH/test/algebra/2167.json
MATH/test/algebra/791.json
MATH/test/algebra/2022.json
MATH/test/algebra/1527.json
MATH/test/algebra/512.json
MATH/test/algebra/142.json
MATH/test/algebra/841.json
MATH/test/algebra/904.json
MATH/test/algebra/1462.json
MATH/test/algebra/1032.json
MATH/test/algebra/338.json
MATH/test/algebra/1248.json
MATH/test/algebra/1862.json
MATH/test/algebra/441.json
MATH/test/algebra/2576.json
MATH/test/algebra/2126.json
MATH/test/algebra/2433.json
MATH/test/algebra/945.json
MATH/test/algebra/416.json
MATH/test/algebra/1423.json
MATH/test/algebra/1264.json
MATH/test/algebra/251.json
MATH/test/algebra/1634.json
MATH/test/algebra/314.json
MATH/test/algebra/744.json
MATH/test/algebra/2274.json
MATH/test/algebra/1458.json
MATH/te

In [29]:
import os
import json

def traverse_and_parse_json(path):
    sub_list = []      # 存储子目录列表
    total_list = []

    # 遍历给定目录
    for sub_path in os.listdir(path):
        full_sub_path = os.path.join(path, sub_path)

        sub_sub_list = []  # 存储解析后的 JSON 数

        # 检查是否为目录
        if os.path.isdir(full_sub_path):

            # 遍历子目录中的文件
            for file_name in os.listdir(full_sub_path):
                if file_name.endswith('.json'):
                    json_file_path = os.path.join(full_sub_path, file_name)

                    # 读取并解析 JSON 文件
                    with open(json_file_path, 'r') as json_file:
                        json_data = json.load(json_file)
                        sub_sub_list.append(json_data)
                        total_list.append(json_data)
            sub_list.append(sub_sub_list)

    return sub_list, sub_sub_list, total_list

# 示例用法
# path = '/content/MATH/train'
sub_list, sub_sub_list, train_total_list = traverse_and_parse_json('/content/MATH/train')
sub_list, sub_sub_list, test_total_list = traverse_and_parse_json('/content/MATH/test')

print("子目录列表:", len(sub_list))
for data in sub_list:
  print("解析后的 JSON 数据:", len(data))

子目录列表: 7
解析后的 JSON 数据: 1187
解析后的 JSON 数据: 903
解析后的 JSON 数据: 540
解析后的 JSON 数据: 546
解析后的 JSON 数据: 474
解析后的 JSON 数据: 479
解析后的 JSON 数据: 871


In [30]:
def list_to_dict(list):
    dict = {'prompt':[], 'completions':[], 'labels': [], 'type': [], 'is_step': [], 'is_end': []}
    for item in list:
        dict['prompt'].append(item['problem'])
        dict['completions'].append([item['solution']]) # string to list
        dict['labels'].append([True])
        dict['type'].append('MATH')
        dict['is_step'].append(False)
        dict['is_end'].append(True)
    return dict

In [31]:

hf_dataset_train = Dataset.from_dict(list_to_dict(train_total_list))
print(hf_dataset_train)
hf_dataset_test = Dataset.from_dict(list_to_dict(test_total_list))
print(hf_dataset_test)
math_dataset = DatasetDict( {'train' : hf_dataset_train,
                'test' : hf_dataset_test})
print(math_dataset)

Dataset({
    features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
    num_rows: 7500
})
Dataset({
    features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
    num_rows: 5000
})
DatasetDict({
    train: Dataset({
        features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
        num_rows: 7500
    })
    test: Dataset({
        features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
        num_rows: 5000
    })
})


In [32]:
from datasets import concatenate_datasets

# 合并数据集
math_dataset['train'] = concatenate_datasets([prm800k_dataset['train'], gsm8k_dataset['train'], math_dataset['train']])
math_dataset['test'] = concatenate_datasets([prm800k_dataset['test'], gsm8k_dataset['test'], math_dataset['test']])

In [33]:
print(math_dataset)

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
        num_rows: 75477
    })
    test: Dataset({
        features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
        num_rows: 7546
    })
})


# PRM Dataset

##PRM800K

In [34]:
from datasets import load_dataset, DatasetDict, Dataset

dataset = load_dataset("plaguss/prm_800k_trl",)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completions', 'labels', 'index'],
        num_rows: 389725
    })
    test: Dataset({
        features: ['prompt', 'completions', 'labels', 'index'],
        num_rows: 10246
    })
})


In [35]:
# STEP SFT时要去除错误STEP的训练数据
# PRM需要有False的数据
def filter_prm(dataset):
    data_list = []
    for data in dataset:
          l = len(data['completions'])
          answer_last_1_step = False
          answer_last_2_step = False
          if '\n\n# Answer\n\n' in data['completions'][-1]:
              answer_last_1_step = True
              # print('last step == \n')
          if l >= 2 and '\n\n# Answer\n\n' in data['completions'][-2]:
              # print('last 2 step == \n')
              answer_last_2_step = True
              # data['completions'] = data['completions'][:-1]
              # data['labels'] = data['labels'][:-1]

          data['is_step'] = True
          data['type'] = 'PRM800K'
          if (answer_last_1_step or  answer_last_2_step): # complemention
              data['is_end'] = True
              data_list.append(data)
          else:
              data['is_end'] = False
              data_list.append(data)
    print(len(data_list))

    # filter dupulation prompt
    data_only_list = []
    pre_string = 'null'
    is_new_prompt = True
    for data in data_list:
        if data['prompt'] == pre_string:
            continue
        else:
            data_only_list.append(data)
            pre_string = data['prompt']
    len(data_only_list)

    return data_only_list

In [36]:
new_dataset_train = filter_prm(dataset['train'])
new_dataset_test = filter_prm(dataset['test'])
print(len(new_dataset_train))
print(len(new_dataset_test))

389725
10246
81544
1647


In [37]:
def list_to_dict(list):
    dict = {'prompt':[], 'completions':[], 'labels': [], 'type': [], 'is_step': [], 'is_end': []}
    for item in list:
        dict['prompt'].append(item['prompt'])
        dict['completions'].append(item['completions'])
        dict['labels'].append(item['labels'])
        dict['type'].append(item['type'])
        dict['is_step'].append(item['is_step'])
        dict['is_end'].append(item['is_end'])
    return dict

In [38]:
hf_dataset_train = Dataset.from_dict(list_to_dict(new_dataset_train))
print(hf_dataset_train)
hf_dataset_test = Dataset.from_dict(list_to_dict(new_dataset_test))
print(hf_dataset_test)
prm800k_dataset = DatasetDict( {'train' : hf_dataset_train,
                'test' : hf_dataset_test})
print(prm800k_dataset)

Dataset({
    features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
    num_rows: 81544
})
Dataset({
    features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
    num_rows: 1647
})
DatasetDict({
    train: Dataset({
        features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
        num_rows: 81544
    })
    test: Dataset({
        features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
        num_rows: 1647
    })
})


## GSM8K

In [39]:
from datasets import load_dataset, DatasetDict, Dataset

dataset = load_dataset("openai/gsm8k",'main')

In [40]:
def process_gsm8k(example):
  text = example['answer']
  lines = text.split('\n')
  labels = [True] * len(lines)
  example['prompt'] = example['question']

  lines[-1] = lines[-1].replace('#### ', 'Answer: ')
  example['completions'] = lines
  example['labels'] = labels


  example['is_end'] = True
  example['is_step'] = True
  example['type'] = 'GSM8K'


  return example

gsm8k_dataset = dataset.map(process_gsm8k, remove_columns=['question', 'answer'])

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

## Qwen/ProcessBench

In [41]:
# Qwen/ProcessBench

from datasets import load_dataset, DatasetDict, Dataset

dataset = load_dataset("Qwen/ProcessBench")
print(dataset)


def process_bench(example):
  steps = example['steps']
  # if example['label'] == -1
  labels = [True] * len(steps)
  if example['label'] != -1:
    if example['final_answer_correct'] == True:
      labels[example['label']] = False
    else:
      for i in range(example['label'], len(steps)):
        labels[i] = False


  example['prompt'] = example['problem']

  example['completions'] = steps
  example['labels'] = labels

  example['is_end'] = True
  example['is_step'] = True
  example['type'] = 'PROCESS_BENCH'

  return example

processbench_dataset = dataset.map(process_bench, remove_columns=['problem', 'steps','final_answer_correct', 'label', 'id', 'generator'])


README.md:   0%|          | 0.00/1.55k [00:00<?, ?B/s]

gsm8k.json:   0%|          | 0.00/574k [00:00<?, ?B/s]

math.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

olympiadbench.json:   0%|          | 0.00/2.81M [00:00<?, ?B/s]

omnimath.json:   0%|          | 0.00/2.84M [00:00<?, ?B/s]

...

DatasetDict({
    gsm8k: Dataset({
        features: ['id', 'generator', 'problem', 'steps', 'final_answer_correct', 'label'],
        num_rows: 400
    })
    math: Dataset({
        features: ['id', 'generator', 'problem', 'steps', 'final_answer_correct', 'label'],
        num_rows: 1000
    })
    olympiadbench: Dataset({
        features: ['id', 'generator', 'problem', 'steps', 'final_answer_correct', 'label'],
        num_rows: 1000
    })
    omnimath: Dataset({
        features: ['id', 'generator', 'problem', 'steps', 'final_answer_correct', 'label'],
        num_rows: 1000
    })
})


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [42]:
from datasets import concatenate_datasets
processbench_dataset_cat =concatenate_datasets([processbench_dataset['gsm8k'],
                                            processbench_dataset['math'],
                                            processbench_dataset['olympiadbench'],
                                            processbench_dataset['omnimath']])

In [43]:
print(processbench_dataset_cat)

Dataset({
    features: ['prompt', 'completions', 'labels', 'is_end', 'is_step', 'type'],
    num_rows: 3400
})


## MATH

In [46]:
def list_to_dict(list):
    dict = {'prompt':[], 'completions':[], 'labels': [], 'type': [], 'is_step': [], 'is_end': []}
    for item in list:
        dict['prompt'].append(item['problem'])
        dict['completions'].append([item['solution']]) # string to list
        dict['labels'].append([True])
        dict['type'].append('MATH')
        dict['is_step'].append(False)
        dict['is_end'].append(True)
    return dict

In [47]:
# Math
hf_dataset_train = Dataset.from_dict(list_to_dict(train_total_list))
print(hf_dataset_train)
hf_dataset_test = Dataset.from_dict(list_to_dict(test_total_list))
print(hf_dataset_test)
math_dataset = DatasetDict( {'train' : hf_dataset_train,
                'test' : hf_dataset_test})
print(math_dataset)

Dataset({
    features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
    num_rows: 7500
})
Dataset({
    features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
    num_rows: 5000
})
DatasetDict({
    train: Dataset({
        features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
        num_rows: 7500
    })
    test: Dataset({
        features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
        num_rows: 5000
    })
})


## Merge

In [48]:
print(prm800k_dataset['train'])
print(gsm8k_dataset['train'])
print(math_dataset['train'])
print(processbench_dataset_cat)

Dataset({
    features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
    num_rows: 81544
})
Dataset({
    features: ['prompt', 'completions', 'labels', 'is_end', 'is_step', 'type'],
    num_rows: 7473
})
Dataset({
    features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
    num_rows: 7500
})
Dataset({
    features: ['prompt', 'completions', 'labels', 'is_end', 'is_step', 'type'],
    num_rows: 3400
})


In [49]:
def stats_label(dataset):
  total = 0
  sum = 0
  for labels in dataset:
    total += len(labels)
    for i in labels:
      sum += i

  print(total)
  print(sum)


stats_label(processbench_dataset_cat['labels'])
stats_label(prm800k_dataset['train']['labels'])
stats_label(gsm8k_dataset['train']['labels'])
stats_label(math_dataset['train']['labels'])

25697
16505
521146
439818
34197
34197
7500
7500


In [50]:
from datasets import concatenate_datasets


# 合并数据集
math_dataset['train'] = concatenate_datasets([prm800k_dataset['train'], gsm8k_dataset['train'], math_dataset['train'], processbench_dataset_cat ])
math_dataset['test'] = concatenate_datasets([prm800k_dataset['test'], gsm8k_dataset['test'], math_dataset['test']] )

In [51]:
print(math_dataset)

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
        num_rows: 99917
    })
    test: Dataset({
        features: ['prompt', 'completions', 'labels', 'type', 'is_step', 'is_end'],
        num_rows: 7966
    })
})


In [ ]:
math_dataset.push_to_hub('you_hf_name/you_dataset_name')